# 1kg_eur — 5 % MAF GRM (Batch)

Builds a genomic relatedness matrix restricted to common variants (MAF > 5 %).
Input: the MAF > 1 % QC-filtered BED produced by `04_grm_panel_qc.ipynb`.

**Step 1** — single Batch task downloads the 1 %-BED, applies `--maf 0.05`, recomputes allele frequencies, uploads the result.

**Step 2** — sharded Batch jobs compute the GRM in parallel (same approach as `06_grm_shards.ipynb`).

## Config

In [ ]:
import math, os, subprocess

PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS  = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS   = f"{WS_GS}/1kg_eur"

# ── input: 1 % MAF BED from notebook 04 ────────────────────────────────────────
BED_NAME     = "1kg_CEUGBR_GRM_QC"
GRM_INPUT_GS = f"{R_GS}/03_grm/grm_input"
PLINK_BIN_GS = f"{R_GS}/03_grm/bin/plink"   # plink 1.9 staged in 06_grm_shards

# ── output: 5 % MAF BED and GRM shards ─────────────────────────────────────────
BED_5PCT_NAME = "1kg_CEUGBR_GRM_5pct"
FILTER_OUT_GS = f"{R_GS}/03_grm/grm_input_5pct"   # filtered BED + .frq
SHARD_5PCT_GS = f"{R_GS}/03_grm/shards_5pct"
LOG_GS        = f"{R_GS}/03_grm/logs_5pct"

# ── filter job sizing ───────────────────────────────────────────────────────────
FILTER_MACHINE = "n1-standard-8"   # 30 GB RAM; plink 1.9 reads BED in chunks
FILTER_DISK_GB = 400               # 94 GB in + ~65 GB out + working space

# ── GRM shard sizing ────────────────────────────────────────────────────────────
# Update BED_SIZE_5PCT_GB after step 1 completes:
#   gcloud storage ls -l {FILTER_OUT_GS}/{BED_5PCT_NAME}.bed | awk '{{printf "%.1f GB\\n", $1/1024/1024/1024}}'
BED_SIZE_5PCT_GB = 70   # estimated; ~75 % of the 94 GB 1 %-BED
SHARD_VCPUS  = 16
_raw_mb = int((BED_SIZE_5PCT_GB * 2 + 4) * 1024)
MEMORY_MB = math.ceil(_raw_mb / 256) * 256
_mem_per_vcpu = MEMORY_MB // SHARD_VCPUS
SHARD_MACHINE = (f"n1-custom-{SHARD_VCPUS}-{MEMORY_MB}-ext"
                 if _mem_per_vcpu > 8192
                 else f"n1-custom-{SHARD_VCPUS}-{MEMORY_MB}")
PLINK_MEM_MB  = MEMORY_MB - 8192
SHARD_DISK_GB = 200

N_SHARDS = 20
N_TASKS  = 5

for k, v in dict(
    BED_5PCT_NAME=BED_5PCT_NAME,
    FILTER_OUT_GS=FILTER_OUT_GS,
    SHARD_5PCT_GS=SHARD_5PCT_GS,
    SHARD_MACHINE=SHARD_MACHINE,
    MEMORY_MB=MEMORY_MB,
    PLINK_MEM_MB=PLINK_MEM_MB,
    N_SHARDS=N_SHARDS,
    N_TASKS=N_TASKS,
).items():
    print(f"  {k}: {v}")

## Install dsub

In [ ]:
subprocess.run(["bash", "-c", f"""
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"
echo "dsub: $(dsub --version)"
echo "wrapper image: $(grep CLOUD_SDK_IMAGE $DSUB_DIR/providers/google_utils.py)"
"""], check=True)

## Step 1: Filter panel to MAF > 5 %

Single Batch task:
1. Downloads the 1 %-filtered BED from `grm_input/` (~94 GB).
2. Applies `plink --maf 0.05 --make-bed` — allele frequencies computed from the round-2 cohort samples.
3. Recomputes `.frq` frequencies on the 5 %-panel for use by the GRM shard jobs.
4. Uploads BED + freqs to `grm_input_5pct/`.

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "$PROJECT_ID" --regions "$REGION" \
  --logging "{LOG_GS}" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" --subnetwork "$SUBNETWORK" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "grm-5pct-filter" \
  --machine-type "{FILTER_MACHINE}" --disk-size "{FILTER_DISK_GB}" \
  --input        PLINK="{PLINK_BIN_GS}" \
  --input-recursive  BED_IN="{GRM_INPUT_GS}" \
  --output-recursive BED_OUT="{FILTER_OUT_GS}" \
  --command '
    set -eo pipefail
    chmod +x "$PLINK"

    # Filter to MAF > 5 % (frequencies computed from the round-2 samples in .fam)
    "$PLINK" \
      --bfile "$BED_IN/{BED_NAME}" \
      --maf 0.05 --make-bed \
      --threads $(nproc) \
      --out "$BED_OUT/{BED_5PCT_NAME}"

    # Allele frequencies for GRM shards (plink 1.9 .frq format)
    "$PLINK" \
      --bfile "$BED_OUT/{BED_5PCT_NAME}" \
      --freq \
      --out "$BED_OUT/{BED_5PCT_NAME}_freq"

    echo "variants after MAF>5%: $(wc -l < \"$BED_OUT/{BED_5PCT_NAME}.bim\")"
    echo "samples: $(wc -l < \"$BED_OUT/{BED_5PCT_NAME}.fam\")"
  ' 2>&1 | tee /tmp/filter_job.log
FILTER_JOB=$(tail -1 /tmp/filter_job.log)
echo "filter job: $FILTER_JOB" | tee /tmp/filter_job_id.txt
"""], check=True)

## Monitor filter job

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "$PROJECT_ID" --location "$REGION" \
  --jobs "grm-5pct-filter*" --status '*' --full
"""], check=False)

## Preflight check

Run after step 1 completes. Check variant and sample counts, update `BED_SIZE_5PCT_GB` in config if the actual size differs from the estimate.

In [ ]:
subprocess.run(["bash", "-c", f"""
echo "=== 5 % panel files ==="
for ext in bed bim fam; do
  gcloud storage ls -l "{FILTER_OUT_GS}/{BED_5PCT_NAME}.$ext" 2>/dev/null \
    || echo "  MISSING: {BED_5PCT_NAME}.$ext"
done

echo
echo "=== frequencies ==="
gcloud storage ls -l "{FILTER_OUT_GS}/{BED_5PCT_NAME}_freq.frq" 2>/dev/null \
  || echo "  MISSING — step 1 must complete first"

echo
echo "=== variant count ==="
gcloud storage cat "{FILTER_OUT_GS}/{BED_5PCT_NAME}.bim" 2>/dev/null | wc -l \
  || echo "  (could not read .bim)"

echo
echo "=== sample count ==="
gcloud storage cat "{FILTER_OUT_GS}/{BED_5PCT_NAME}.fam" 2>/dev/null | wc -l \
  || echo "  (could not read .fam)"

echo
echo "=== BED size (update BED_SIZE_5PCT_GB in config if different from 70) ==="
gcloud storage ls -l "{FILTER_OUT_GS}/{BED_5PCT_NAME}.bed" 2>/dev/null \
  | awk '{{printf "  %.1f GB\\n", $1/1024/1024/1024}}' || true

echo
echo "=== plink 1.9 binary ==="
gcloud storage ls -l "{PLINK_BIN_GS}" 2>/dev/null || echo "  not staged — run 06_grm_shards first"
"""], check=True)

## Step 2: Build GRM shard tasks

In [ ]:
shards_per_task = N_SHARDS // N_TASKS
tasks_tsv = "/tmp/grm_5pct_tasks.tsv"
with open(tasks_tsv, "w") as fh:
    fh.write("--env TASK_SHARDS\n")
    for t in range(N_TASKS):
        shards = list(range(t * shards_per_task + 1,
                             (t + 1) * shards_per_task + 1))
        fh.write(",".join(map(str, shards)) + "\n")
print(f"tasks written: {tasks_tsv}")
print(open(tasks_tsv).read())

## Submit GRM shard jobs

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "$PROJECT_ID" --regions "$REGION" \
  --logging "{LOG_GS}" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" --subnetwork "$SUBNETWORK" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "grm-5pct-shards" \
  --machine-type "{SHARD_MACHINE}" --disk-size "{SHARD_DISK_GB}" \
  --input        PLINK_BIN="{PLINK_BIN_GS}" \
  --input-recursive  BED_DIR="{FILTER_OUT_GS}" \
  --output-recursive SHARD_DIR="{SHARD_5PCT_GS}" \
  --env BED_NAME="{BED_5PCT_NAME}" \
  --env N_SHARDS="{N_SHARDS}" \
  --env PLINK_MEM_MB="{PLINK_MEM_MB}" \
  --tasks /tmp/grm_5pct_tasks.tsv \
  --command '
    set -eo pipefail
    chmod +x "$PLINK_BIN"
    BED_PREFIX="$BED_DIR/$BED_NAME"
    FREQ_PATH="$BED_DIR/${{BED_NAME}}_freq.frq"
    IFS="," read -ra SHARDS <<< "$TASK_SHARDS"
    for k in "${{SHARDS[@]}}"; do
      echo "--- shard $k / $N_SHARDS ---"
      "$PLINK_BIN" \
        --bfile "$BED_PREFIX" \
        --read-freq "$FREQ_PATH" \
        --make-grm-bin --parallel "$k" "$N_SHARDS" \
        --memory "$PLINK_MEM_MB" \
        --out "$SHARD_DIR/grm.shard$k"
      rm -f "$SHARD_DIR/grm.shard$k.grm.N.bin"
    done
  ' 2>&1 | tee /tmp/grm_5pct_jobs.log
cat /tmp/grm_5pct_jobs.log | tee /tmp/grm_5pct_job_id.txt
"""], check=True)

## Monitor shard jobs

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "$PROJECT_ID" --location "$REGION" \
  --jobs "grm-5pct-shards*" --status '*' --full
"""], check=False)

## Copy notebook to bucket

In [ ]:
import subprocess, os
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/grm_5pct_shards.ipynb')
_gs = f'{SHARD_5PCT_GS}/notebooks/grm_5pct_shards.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')